In [6]:
# # huggingface에서 klue/bert-base 모델 가져오기
# import torch
# from transformers import AutoTokenizer, AutoModelForMaskedLM

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model_name = "klue/bert-base"
# 모델이 너무 무거워 경량화 모델로 변경

# huggingface에서 klue/roberta-small 모델 가져오기
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "klue/roberta-small"


In [17]:
import datasets
print(datasets.__file__)


AttributeError: partially initialized module 'datasets' has no attribute 'utils' (most likely due to a circular import)

In [16]:
# huggingface에서 KOLD 데이터 가져오기
from datasets import load_dataset

ds = load_dataset("nayohan/KOLD", cache_dir="./dataset_cache")


AttributeError: partially initialized module 'datasets' has no attribute 'utils' (most likely due to a circular import)

In [4]:
# 데이터 확인하기
ds

DatasetDict({
    train: Dataset({
        features: ['guid', 'source', 'date', 'title', 'comment', 'OFF', 'TGT', 'GRP', 'OFF_span', 'TGT_span', 'raw_labels'],
        num_rows: 40429
    })
})

In [5]:
# 토큰화 함수 지정 및 라벨링
def encode_example(example):
    text = example["comment"]

    def safe_int_list(raw):
        return [int(i) for i in raw if isinstance(i, (int, float)) or (isinstance(i, str) and i.strip().isdigit())]

    off_span = safe_int_list(example["OFF_span"])
    tgt_span = safe_int_list(example["TGT_span"])

    tokens = tokenizer(text, truncation=True, padding='max_length', max_length=128, return_offsets_mapping=True)
    labels = [0] * len(tokens["input_ids"])

    for idx in off_span:
        if idx < len(labels):
            labels[idx] = 1  # offensive

    for idx in tgt_span:
        if idx < len(labels):
            if labels[idx] == 1:
                labels[idx] = 3  # offensive + target
            elif labels[idx] == 0:
                labels[idx] = 2  # target

    tokens["labels"] = labels
    return tokens


In [6]:
# 토크나이저 및 모델 설정
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name,  num_labels=4)


In [21]:
# 인코딩
encoded = ds["train"].map(
    encode_example,
    remove_columns=ds["train"].column_names,
    num_proc=2
)

# 데이터를 RAM에 한 번에 올리는 코드, RAM 뻑나서 주석처리
# encoded.set_format("torch")
split = encoded.train_test_split(test_size=0.2, seed=42)


Map (num_proc=2):   0%|          | 0/40429 [00:00<?, ? examples/s]

In [22]:
# 모델 확인
model

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm)

In [23]:
# 데이터셋 전처리
encoded_ds = ds["train"].map(encode_example, remove_columns=ds["train"].column_names)
encoded_ds.set_format("torch")
train_test = encoded_ds.train_test_split(test_size=0.2, seed=42)


In [24]:
# 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# 학습 설정
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/kold-model",
    eval_strategy="steps",
    save_steps=1000,
    eval_steps=500,
    logging_steps=200,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="/content/drive/MyDrive/kold-logs",
    save_total_limit=2,
    fp16=True,
    report_to="none",
)


In [27]:
# 정확도 계산을 위한 코드
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids.flatten()
    preds = np.argmax(pred.predictions, axis=2).flatten()

    # 마스킹된 토큰(-100) 제외
    valid = labels != -100
    labels = labels[valid]
    preds = preds[valid]

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [28]:
# Trainer 구성
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    # 10000개 뻑남, 1000개 부터 점진적으로 증가 예정
    train_dataset=split["train"].shuffle(seed=42).select(range(1000)),
    eval_dataset=split["test"],
    compute_metrics=compute_metrics,
)


In [29]:
# api key 무시
import os
os.environ["WANDB_DISABLED"] = "true"


In [30]:
# 학습
trainer.train()


Step,Training Loss,Validation Loss


TrainOutput(global_step=62, training_loss=4.877190374558972, metrics={'train_runtime': 1070.4117, 'train_samples_per_second': 0.934, 'train_steps_per_second': 0.058, 'total_flos': 32876292734976.0, 'train_loss': 4.877190374558972, 'epoch': 0.992})

In [31]:
# 학습한 모델 저장
model.save_pretrained("./my_trained_model")
tokenizer.save_pretrained("./my_trained_model")


('./my_trained_model/tokenizer_config.json',
 './my_trained_model/special_tokens_map.json',
 './my_trained_model/vocab.txt',
 './my_trained_model/added_tokens.json',
 './my_trained_model/tokenizer.json')

In [32]:
# github 저장소 clone
!git config --global user.name "sosomeet"
!git config --global user.email "skcmemformlf0@gmail.com"  # GitHub 등록된 이메일

# 저장소 클론
!git clone https://github.com/sosomeet/TM_soft_hate_classifier.git


Cloning into 'TM_soft_hate_classifier'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 9 (delta 0), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 5.78 MiB | 9.78 MiB/s, done.


In [33]:
# 모델 디렉토리 복사
!cp -r ./my_trained_model TM_soft_hate_classifier/model


In [34]:
# github에 push
%cd TM_soft_hate_classifier
!git add .
!git commit -m "Add trained model"


/content/TM_soft_hate_classifier
[main 5c68d21] Add trained model
 6 files changed, 64301 insertions(+)
 create mode 100644 model/config.json
 create mode 100644 model/model.safetensors
 create mode 100644 model/special_tokens_map.json
 create mode 100644 model/tokenizer.json
 create mode 100644 model/tokenizer_config.json
 create mode 100644 model/vocab.txt


In [35]:
# 예측
def predict_tokens(text):
    # 입력 문장 토큰화
    tokens = tokenizer(text, return_tensors="pt", truncation=True, padding='max_length', max_length=128)
    output = model(**tokens)
    predictions = output.logits.argmax(dim=-1).squeeze().tolist()

    input_ids = tokens["input_ids"].squeeze().tolist()
    token_strs = tokenizer.convert_ids_to_tokens(input_ids)

    result = []
    for token, label in zip(token_strs, predictions):
        result.append((token, label))

    return result


In [36]:
# 예측 시각화
def visualize_prediction(text):
    label_map = {
        0: "🟢 normal",
        1: "🔴 offensive",
        2: "🟡 target",
        3: "🔵 off+target"
    }

    pred = predict_tokens(text)
    for token, label in pred:
        if token in ["[CLS]", "[PAD]", "[SEP]"]:
            continue
        print(f"{token:15} → {label_map.get(label, label)}")


In [37]:
# 실행 예시
visualize_prediction("이슬람 믿는 나라는 다 그래")


이슬람             → 🟢 normal
믿               → 🟢 normal
##는             → 🟢 normal
나라              → 🟢 normal
##는             → 🟢 normal
다               → 🟢 normal
그래              → 🟢 normal


In [38]:
visualize_prediction("우리나라는 살기 좋은 나라인 것 같아")


우리나라            → 🟢 normal
##는             → 🟢 normal
살               → 🟢 normal
##기             → 🟢 normal
좋               → 🟢 normal
##은             → 🟢 normal
나라              → 🟢 normal
##인             → 🟢 normal
것               → 🟢 normal
같               → 🟢 normal
##아             → 🟢 normal


In [39]:
visualize_prediction("나쁜 쓰레기 이슬람 교도")


나쁜              → 🟢 normal
쓰레기             → 🟢 normal
이슬람             → 🟢 normal
교도              → 🟢 normal


In [ ]:
results = trainer.evaluate()
print(results)